# Data Preparation and Cleaning

This notebook loads the IMD dataset, filters for Leeds, and cleans the data.

In [1]:
# Configuration
INPUT_FILE = 'File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv'
LOCAL_AUTHORITY = 'Leeds'
OUTPUT_FOLDER = 'output'

import pandas as pd
import numpy as np
import os

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("=" * 60)
print("DATA PREPARATION AND CLEANING")
print("=" * 60)

DATA PREPARATION AND CLEANING


## Load IMD Data

In [2]:
print(f"Loading IMD data from {INPUT_FILE}...")

# Try different encodings
encodings = ['utf-8', 'latin-1', 'cp1252']
df = None

for encoding in encodings:
    try:
        df = pd.read_csv(INPUT_FILE, encoding=encoding, low_memory=False)
        print(f"Successfully loaded data with {encoding} encoding")
        break
    except (UnicodeDecodeError, FileNotFoundError):
        continue

if df is None:
    raise ValueError("Could not read file with any encoding")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head()

Loading IMD data from File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv...
Successfully loaded data with utf-8 encoding
Dataset shape: (32844, 57)
Columns: 57


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,...,16364,5,1.503,1615,1,1296,175,656,465,715
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,...,22676,7,1.196,2969,1,1156,182,580,394,620
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,...,17318,6,2.207,162,1,1350,146,759,445,804
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,...,25218,8,1.769,849,1,1121,229,692,200,683
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.837,14486,5,0.117,14023,5,...,14745,5,0.969,4368,2,2040,522,1297,221,1285


## Filter for Leeds

In [3]:
print(f"\nFiltering data for {LOCAL_AUTHORITY}...")

# Find Local Authority column
la_col = None
possible_names = [
    'Local Authority District name (2019)',
    'Local Authority District (2019)',
    'Local Authority',
    'LA name',
    'LAD19NM'
]

for name in possible_names:
    if name in df.columns:
        la_col = name
        break

if la_col is None:
    print("Available columns:", df.columns.tolist()[:10])
    raise ValueError("Could not find Local Authority column")

print(f"Using column: {la_col}")

# Filter for Leeds
leeds_df = df[df[la_col].str.contains(LOCAL_AUTHORITY, case=False, na=False)].copy()

print(f"Found {len(leeds_df)} LSOAs in {LOCAL_AUTHORITY}")
print(f"Original dataset had {len(df)} LSOAs")
leeds_df.head()


Filtering data for Leeds...
Using column: Local Authority District name (2019)
Found 482 LSOAs in Leeds
Original dataset had 32844 LSOAs


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
10947,E01011264,Leeds 011A,E08000035,Leeds,20.525,13915,5,0.088,18238,6,...,5201,2,0.054,14415,5,1252,188,653,411,665
10948,E01011265,Leeds 009A,E08000035,Leeds,13.602,20368,7,0.080,19551,6,...,4494,2,-0.114,17248,6,1788,321,999,468,1007
10949,E01011266,Leeds 008A,E08000035,Leeds,5.808,29666,10,0.021,32142,10,...,10528,4,-0.194,18531,6,2381,483,1431,467,1444
10950,E01011267,Leeds 009B,E08000035,Leeds,27.863,9111,3,0.153,10261,4,...,6552,2,-0.100,16979,6,1489,316,780,393,783
10951,E01011268,Leeds 010A,E08000035,Leeds,34.444,6082,2,0.161,9579,3,...,3716,2,0.078,14035,5,1358,270,823,265,824


## Check for Missing Values

In [4]:
print("\nChecking for missing values...")

key_columns = ['IMD Score', 'Health Score', 'Population']
for col in key_columns:
    matching_cols = [c for c in leeds_df.columns if col.lower() in c.lower() or c.lower() in col.lower()]
    if matching_cols:
        actual_col = matching_cols[0]
        missing_count = leeds_df[actual_col].isna().sum()
        missing_pct = (missing_count / len(leeds_df)) * 100
        print(f"  {actual_col}: {missing_count} missing ({missing_pct:.2f}%)")


Checking for missing values...
  Total population: mid 2015 (excluding prisoners): 0 missing (0.00%)


## Clean Data

In [5]:
print("\nCleaning data...")

# Remove completely empty rows
leeds_df = leeds_df.dropna(how='all')

# Remove duplicates
initial_rows = len(leeds_df)
leeds_df = leeds_df.drop_duplicates()
duplicates_removed = initial_rows - len(leeds_df)

if duplicates_removed > 0:
    print(f"  Removed {duplicates_removed} duplicate rows")

# Ensure correct data types
numeric_patterns = ['Score', 'Rank', 'Decile', 'Population', 'Count', 'Rate', 'Percent']
for col in leeds_df.columns:
    col_lower = col.lower()
    if any(pattern.lower() in col_lower for pattern in numeric_patterns):
        if 'code' not in col_lower and 'name' not in col_lower:
            leeds_df[col] = pd.to_numeric(leeds_df[col], errors='coerce')

print("Data types corrected")
print(f"\nFinal dataset shape: {leeds_df.shape}")
print(f"Final dataset columns: {len(leeds_df.columns)}")


Cleaning data...
Data types corrected

Final dataset shape: (482, 57)
Final dataset columns: 57


## Save Cleaned Data

In [6]:
output_path = os.path.join(OUTPUT_FOLDER, 'Leeds_IMD_LSOA.csv')
leeds_df.to_csv(output_path, index=False)
print(f"\nSaved {len(leeds_df)} rows to {output_path}")

print("\n" + "=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)

leeds_df.head()


Saved 482 rows to output\Leeds_IMD_LSOA.csv

DATA PREPARATION COMPLETE


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
10947,E01011264,Leeds 011A,E08000035,Leeds,20.525,13915,5,0.088,18238,6,...,5201,2,0.054,14415,5,1252,188,653,411,665
10948,E01011265,Leeds 009A,E08000035,Leeds,13.602,20368,7,0.080,19551,6,...,4494,2,-0.114,17248,6,1788,321,999,468,1007
10949,E01011266,Leeds 008A,E08000035,Leeds,5.808,29666,10,0.021,32142,10,...,10528,4,-0.194,18531,6,2381,483,1431,467,1444
10950,E01011267,Leeds 009B,E08000035,Leeds,27.863,9111,3,0.153,10261,4,...,6552,2,-0.100,16979,6,1489,316,780,393,783
10951,E01011268,Leeds 010A,E08000035,Leeds,34.444,6082,2,0.161,9579,3,...,3716,2,0.078,14035,5,1358,270,823,265,824
